# CuKD-XAI: WSN-DS Deployment + QAT Proof Route
## v2.3-derived

**Author:** Nishant Harkut (2023IMG-040), ABV-IIITM Gwalior

**Upload WSN-DS.csv before running.**

## Install dependencies

In [1]:
# Optional packages used by this runtime-only route:
# !pip install -q onnx onnxruntime openvino

## Imports and configuration

In [2]:
import importlib
import json
import os
import platform
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

EXISTING_DEPLOYMENT_OUTPUT_DIR = os.environ.get(
    "EXISTING_DEPLOYMENT_OUTPUT_DIR",
    str(Path("..") / "wsnds_deployment_qat_outputs"),
)
RUNTIME_OUTPUT_DIR = os.environ.get("RUNTIME_OUTPUT_DIR", "")
WSNDS_PATH = os.environ.get("WSNDS_PATH", str(Path("..") / "WSN-DS.csv"))
INSTALL_OPTIONAL_DEPLOYMENT_DEPS = True
ENABLE_ONNX_BENCHMARKS = True
ENABLE_OPENVINO_BENCHMARKS = True
ONNX_OPSET_VERSION = 17
LATENCY_WARMUP = 50
LATENCY_RUNS_B1 = 1000
LATENCY_RUNS_B64 = 300
OPENVINO_PARITY_MIN_AGREEMENT = 0.995
OPENVINO_PARITY_MAX_MACRO_F1_DELTA = 0.01

STUDENT_A_HIDDEN = (32, 16)
STUDENT_B_HIDDEN = (64, 32)
NUM_CLASSES_EXPECTED = 5
INPUT_DIM_EXPECTED = 17

MODEL_ARTIFACTS = {
    "D_student_A_scratch": {
        "hidden": STUDENT_A_HIDDEN,
        "artifact": "D_student_A_scratch_fp32.pt",
    },
    "E_student_A_KD_from_RF": {
        "hidden": STUDENT_A_HIDDEN,
        "artifact": "E_student_A_KD_from_RF_fp32.pt",
    },
    "J_student_A_CoDistill_RF_CL": {
        "hidden": STUDENT_A_HIDDEN,
        "artifact": "J_student_A_CoDistill_RF_CL_fp32.pt",
    },
    "D_student_B_scratch": {
        "hidden": STUDENT_B_HIDDEN,
        "artifact": "D_student_B_scratch_fp32.pt",
    },
    "E_student_B_KD_from_RF": {
        "hidden": STUDENT_B_HIDDEN,
        "artifact": "E_student_B_KD_from_RF_fp32.pt",
    },
    "J_student_B_CoDistill_RF_CL": {
        "hidden": STUDENT_B_HIDDEN,
        "artifact": "J_student_B_CoDistill_RF_CL_fp32.pt",
    },
}
DEPLOYMENT_BENCHMARK_MODELS = list(MODEL_ARTIFACTS.keys())

print("Runtime-only benchmark from existing deployment artifacts")
print(f"Existing output dir setting: {EXISTING_DEPLOYMENT_OUTPUT_DIR}")
print(f"WSNDS_PATH setting: {WSNDS_PATH}")

Runtime-only benchmark from existing deployment artifacts
Existing output dir setting: ..\wsnds_deployment_qat_outputs
WSNDS_PATH setting: ..\WSN-DS.csv


## Model definition and path resolution

In [3]:
class StudentMLP(nn.Module):
    """v2.3 student MLP. No BatchNorm, Linear-ReLU stack."""
    def __init__(self, input_dim: int = 17, hidden_dims: tuple = (32, 16),
                 num_classes: int = 5):
        super().__init__()
        layers = []
        prev = input_dim
        for hidden in hidden_dims:
            layers.append(nn.Linear(prev, hidden))
            layers.append(nn.ReLU())
            prev = hidden
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def _unique_paths(paths):
    unique = []
    seen = set()
    for path in paths:
        try:
            key = str(path.resolve())
        except OSError:
            key = str(path)
        if key not in seen:
            seen.add(key)
            unique.append(path)
    return unique


def _strip_single_parent(path: Path) -> Path:
    parts = path.parts
    if parts and parts[0] == "..":
        return Path(*parts[1:]) if len(parts) > 1 else Path(".")
    return path


def candidate_paths(path_text: str):
    configured = Path(os.path.expanduser(str(path_text)))
    if configured.is_absolute():
        return [configured]
    cwd = Path.cwd()
    stripped = _strip_single_parent(configured)
    candidates = [cwd / configured, cwd / stripped]
    try:
        script_dir = Path(__file__).resolve().parent
        candidates.extend([
            script_dir / configured,
            script_dir / stripped,
            script_dir.parent / stripped,
            script_dir.parent / configured,
        ])
    except NameError:
        pass
    candidates.extend([
        cwd / "Final" / "wsnds_deployment_qat_outputs",
        cwd / "wsnds_deployment_qat_outputs",
    ])
    return _unique_paths(candidates)


def resolve_existing_output_dir(path_text: str = EXISTING_DEPLOYMENT_OUTPUT_DIR) -> Path:
    for candidate in candidate_paths(path_text):
        if candidate.exists() and candidate.is_dir():
            return candidate.resolve()
    searched = "\n".join(str(path) for path in candidate_paths(path_text))
    raise FileNotFoundError(
        "Existing deployment output folder not found. It should contain tmp/*_fp32.pt. "
        f"Set EXISTING_DEPLOYMENT_OUTPUT_DIR if needed. Searched:\n{searched}"
    )


def resolve_file(path_text: str, extra_candidates=None) -> Path:
    candidates = candidate_paths(path_text)
    if extra_candidates:
        candidates.extend(extra_candidates)
    for candidate in _unique_paths(candidates):
        if candidate.exists() and candidate.is_file():
            return candidate.resolve()
    searched = "\n".join(str(path) for path in _unique_paths(candidates))
    raise FileNotFoundError(f"File not found. Searched:\n{searched}")


def validate_required_artifacts(tmp_dir: Path) -> None:
    missing = []
    for spec in MODEL_ARTIFACTS.values():
        artifact_path = tmp_dir / spec["artifact"]
        if not artifact_path.exists():
            missing.append(str(artifact_path))
    if missing:
        missing_text = "\n".join(missing)
        raise FileNotFoundError(
            "Missing required fp32 artifacts. Run or copy completed deployment/QAT outputs first:\n"
            f"{missing_text}"
        )


def runtime_output_dir(existing_output_dir: Path) -> Path:
    if RUNTIME_OUTPUT_DIR:
        out = Path(RUNTIME_OUTPUT_DIR)
        return out if out.is_absolute() else (Path.cwd() / out).resolve()
    return existing_output_dir / "runtime_from_existing_outputs"


existing_output_dir = resolve_existing_output_dir()
existing_tmp_dir = existing_output_dir / "tmp"
validate_required_artifacts(existing_tmp_dir)
output_dir = runtime_output_dir(existing_output_dir)
artifact_dir = output_dir / "deployable_runtime_artifacts"
output_dir.mkdir(parents=True, exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Resolved existing deployment outputs: {existing_output_dir}")
print(f"Runtime-only outputs: {output_dir}")

Resolved existing deployment outputs: C:\Users\btpUser\Desktop\Nishant_2023IMG_wct\CuKD-XAI\Final\wsnds_deployment_qat_outputs
Runtime-only outputs: C:\Users\btpUser\Desktop\Nishant_2023IMG_wct\CuKD-XAI\Final\wsnds_deployment_qat_outputs\runtime_from_existing_outputs


## Load WSN-DS test split only

In [4]:
wsnds_file = resolve_file(
    WSNDS_PATH,
    extra_candidates=[
        existing_output_dir.parent / "WSN-DS.csv",
        existing_output_dir.parent / "Relevant Files" / "WSN-DS.csv",
        Path.cwd() / "WSN-DS.csv",
    ],
)
print(f"Resolved WSN-DS CSV: {wsnds_file}")

df = pd.read_csv(wsnds_file)
df.columns = df.columns.str.strip()
target_candidates = ["Attack type", "Attack_Type", "attack_type", "Attack Type", "class"]
target_col = next((candidate for candidate in target_candidates if candidate in df.columns), df.columns[-1])
for id_col in ["id", "Id", "ID"]:
    if id_col in df.columns:
        df = df.drop(id_col, axis=1)
        break

df[target_col] = df[target_col].astype(str).str.strip()
label_encoder = LabelEncoder()
df[target_col] = label_encoder.fit_transform(df[target_col])
CLASS_NAMES = label_encoder.classes_.tolist()
NUM_CLASSES = len(CLASS_NAMES)
X_all = df.drop(target_col, axis=1).values.astype(np.float32)
y_all = df[target_col].values.astype(np.int64)
INPUT_DIM = X_all.shape[1]
if INPUT_DIM != INPUT_DIM_EXPECTED or NUM_CLASSES != NUM_CLASSES_EXPECTED:
    raise RuntimeError(
        f"Unexpected WSN-DS shape/classes: input_dim={INPUT_DIM}, num_classes={NUM_CLASSES}"
    )
scaler = StandardScaler()
X_all_std = scaler.fit_transform(X_all)
_, X_test_np, _, y_test_np = train_test_split(
    X_all_std, y_all, test_size=0.15, random_state=42, stratify=y_all
)
print(f"Test split: {X_test_np.shape}, classes={CLASS_NAMES}")

Resolved WSN-DS CSV: C:\Users\btpUser\Desktop\Nishant_2023IMG_wct\CuKD-XAI\WSN-DS.csv
Test split: (56200, 17), classes=['Blackhole', 'Flooding', 'Grayhole', 'Normal', 'TDMA']


## Runtime benchmark utilities

In [5]:
def try_import_optional_deployment_modules() -> dict:
    optional_specs = {"onnx": "onnx", "onnxruntime": "onnxruntime", "openvino": "openvino"}
    status = {}
    modules = {}
    for module_name, package_name in optional_specs.items():
        try:
            modules[module_name] = importlib.import_module(module_name)
            status[f"{module_name}_available"] = True
            status[f"{module_name}_error"] = None
            continue
        except Exception as first_exc:
            if INSTALL_OPTIONAL_DEPLOYMENT_DEPS:
                try:
                    print(f"Installing optional deployment package: {package_name}")
                    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
                    modules[module_name] = importlib.import_module(module_name)
                    status[f"{module_name}_available"] = True
                    status[f"{module_name}_error"] = None
                    continue
                except Exception as install_exc:
                    status[f"{module_name}_available"] = False
                    status[f"{module_name}_error"] = f"{type(install_exc).__name__}: {install_exc}"
            else:
                status[f"{module_name}_available"] = False
                status[f"{module_name}_error"] = f"{type(first_exc).__name__}: {first_exc}"
            modules[module_name] = None
    status["onnx_available"] = bool(status.get("onnx_available"))
    status["onnxruntime_available"] = bool(status.get("onnxruntime_available"))
    status["openvino_available"] = bool(status.get("openvino_available"))
    return {"modules": modules, "optional_dependency_status": status}


def load_existing_student_model(model_name: str) -> StudentMLP:
    spec = MODEL_ARTIFACTS[model_name]
    artifact_path = existing_tmp_dir / spec["artifact"]
    if not artifact_path.exists():
        raise FileNotFoundError(f"Missing existing fp32 artifact: {artifact_path}")
    model = StudentMLP(INPUT_DIM, spec["hidden"], NUM_CLASSES)
    state = torch.load(artifact_path, map_location="cpu")
    model.load_state_dict(state)
    model.eval()
    return model


def evaluate_numpy_predictions(preds: np.ndarray, y_true: np.ndarray) -> dict:
    acc = accuracy_score(y_true, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, preds, average="macro", zero_division=0
    )
    return {
        "accuracy": float(acc),
        "macro_precision": float(prec),
        "macro_recall": float(rec),
        "macro_f1": float(f1),
    }


def latency_stats(times_ms: list, batch_size: int) -> dict:
    arr = np.asarray(times_ms, dtype=np.float64)
    mean_ms = float(arr.mean()) if arr.size else 0.0
    return {
        f"latency_mean_ms_b{batch_size}": mean_ms,
        f"latency_std_ms_b{batch_size}": float(arr.std()) if arr.size else 0.0,
        f"latency_p50_ms_b{batch_size}": float(np.percentile(arr, 50)) if arr.size else 0.0,
        f"latency_p95_ms_b{batch_size}": float(np.percentile(arr, 95)) if arr.size else 0.0,
        f"latency_p99_ms_b{batch_size}": float(np.percentile(arr, 99)) if arr.size else 0.0,
        f"throughput_samples_per_s_b{batch_size}": float(batch_size * 1000.0 / mean_ms) if mean_ms > 0 else 0.0,
    }


def export_student_to_onnx(model: nn.Module, output_path: Path) -> float:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    model_cpu = model.cpu().eval()
    dummy = torch.randn(1, INPUT_DIM, dtype=torch.float32)
    torch.onnx.export(
        model_cpu,
        dummy,
        str(output_path),
        input_names=["features"],
        output_names=["logits"],
        dynamic_axes={"features": {0: "batch"}, "logits": {0: "batch"}},
        opset_version=ONNX_OPSET_VERSION,
    )
    return output_path.stat().st_size / 1024


def quantize_onnx_dynamic(onnx_path: Path, quantized_path: Path) -> float:
    from onnxruntime.quantization import QuantType, quantize_dynamic
    quantized_path.parent.mkdir(parents=True, exist_ok=True)
    quantize_dynamic(str(onnx_path), str(quantized_path), weight_type=QuantType.QInt8)
    return quantized_path.stat().st_size / 1024


def onnxruntime_predict(session, X_np: np.ndarray, batch_size: int = 4096) -> np.ndarray:
    input_name = session.get_inputs()[0].name
    preds = []
    for start in range(0, len(X_np), batch_size):
        xb = X_np[start:start + batch_size].astype(np.float32, copy=False)
        logits = session.run(None, {input_name: xb})[0]
        preds.append(np.argmax(logits, axis=1))
    return np.concatenate(preds)


def measure_onnxruntime_latency(session, X_np: np.ndarray, batch_size: int,
                                warmup: int, runs: int) -> dict:
    input_name = session.get_inputs()[0].name
    Xb = X_np[:batch_size].astype(np.float32, copy=False)
    for _ in range(warmup):
        session.run(None, {input_name: Xb})
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        session.run(None, {input_name: Xb})
        times.append((time.perf_counter() - start) * 1000.0)
    return latency_stats(times, batch_size)


def benchmark_onnxruntime_model(onnx_path: Path, X_np: np.ndarray, y_np: np.ndarray,
                                return_preds: bool = False):
    import onnxruntime as ort
    session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    preds = onnxruntime_predict(session, X_np)
    metrics = evaluate_numpy_predictions(preds, y_np)
    result = {
        **metrics,
        **measure_onnxruntime_latency(session, X_np, 1, LATENCY_WARMUP, LATENCY_RUNS_B1),
        **measure_onnxruntime_latency(session, X_np, 64, LATENCY_WARMUP, LATENCY_RUNS_B64),
    }
    if return_preds:
        return result, preds
    return result


def openvino_output_array(infer_result, compiled_model=None):
    if compiled_model is not None:
        try:
            return infer_result[compiled_model.output(0)]
        except Exception:
            pass
    if hasattr(infer_result, "values"):
        values = list(infer_result.values())
        if len(values) == 1:
            return values[0]
    if isinstance(infer_result, (list, tuple)):
        return infer_result[0]
    return infer_result


def openvino_logits_2d(logits, expected_batch: int,
                       expected_classes: int = NUM_CLASSES) -> np.ndarray:
    arr = np.asarray(logits)
    if arr.ndim == 0:
        raise RuntimeError(f"Unexpected OpenVINO scalar logits shape: {arr.shape}")
    if arr.ndim == 1:
        if expected_batch != 1:
            raise RuntimeError(
                f"Unexpected OpenVINO 1D logits for batch {expected_batch}: {arr.shape}"
            )
        arr = arr.reshape(1, -1)
    elif arr.ndim == 2:
        if arr.shape[0] == expected_batch:
            pass
        elif arr.shape[1] == expected_batch:
            arr = arr.T
        elif expected_batch == 1:
            arr = arr.reshape(1, -1)
        else:
            raise RuntimeError(f"Unexpected OpenVINO logits shape: {arr.shape}")
    else:
        batch_axes = [axis for axis, size in enumerate(arr.shape) if size == expected_batch]
        if expected_batch != 1 and len(batch_axes) == 1:
            arr = np.moveaxis(arr, batch_axes[0], 0).reshape(expected_batch, -1)
        elif arr.shape[0] == expected_batch:
            arr = arr.reshape(expected_batch, -1)
        elif expected_batch == 1:
            arr = arr.reshape(1, -1)
        else:
            raise RuntimeError(f"Unexpected OpenVINO logits shape: {arr.shape}")
    if arr.shape != (expected_batch, expected_classes):
        raise RuntimeError(
            f"Unexpected OpenVINO logits shape after normalization: {arr.shape}; "
            f"expected ({expected_batch}, {expected_classes})"
        )
    return arr


def openvino_predict(compiled_model, X_np: np.ndarray, batch_size: int = 4096) -> np.ndarray:
    preds = []
    for start in range(0, len(X_np), batch_size):
        xb = X_np[start:start + batch_size].astype(np.float32, copy=False)
        infer_result = compiled_model([xb])
        logits = openvino_output_array(infer_result, compiled_model)
        logits_2d = openvino_logits_2d(logits, expected_batch=len(xb))
        preds.append(np.argmax(logits_2d, axis=1))
    return np.concatenate(preds)


def validate_openvino_parity(openvino_preds: np.ndarray, reference_preds: np.ndarray,
                             openvino_metrics: dict, reference_metrics: dict) -> dict:
    if reference_preds is None or reference_metrics is None:
        raise RuntimeError("openvino_parity_failed: missing ONNX Runtime FP32 reference")
    if len(openvino_preds) != len(reference_preds):
        raise RuntimeError(
            f"openvino_parity_failed: prediction length mismatch "
            f"openvino={len(openvino_preds)} reference={len(reference_preds)}"
        )
    agreement = float(np.mean(openvino_preds == reference_preds))
    f1_delta = abs(float(openvino_metrics["macro_f1"]) - float(reference_metrics["macro_f1"]))
    if agreement < OPENVINO_PARITY_MIN_AGREEMENT or f1_delta > OPENVINO_PARITY_MAX_MACRO_F1_DELTA:
        raise RuntimeError(
            "openvino_parity_failed: "
            f"agreement={agreement:.6f}, macro_f1_delta={f1_delta:.6f}, "
            f"openvino_macro_f1={openvino_metrics['macro_f1']:.6f}, "
            f"onnx_macro_f1={reference_metrics['macro_f1']:.6f}"
        )
    return {
        "openvino_prediction_agreement_vs_onnx": agreement,
        "openvino_macro_f1_delta_vs_onnx": f1_delta,
    }


def measure_openvino_latency(compiled_model, X_np: np.ndarray, batch_size: int,
                             warmup: int, runs: int) -> dict:
    Xb = X_np[:batch_size].astype(np.float32, copy=False)
    for _ in range(warmup):
        compiled_model([Xb])
    times = []
    for _ in range(runs):
        start = time.perf_counter()
        compiled_model([Xb])
        times.append((time.perf_counter() - start) * 1000.0)
    return latency_stats(times, batch_size)


def benchmark_openvino_model(onnx_path: Path, X_np: np.ndarray, y_np: np.ndarray,
                             reference_preds: np.ndarray = None,
                             reference_metrics: dict = None) -> dict:
    import openvino as ov
    core = ov.Core()
    ov_model = core.read_model(str(onnx_path))
    compiled_model = core.compile_model(ov_model, "CPU")
    preds = openvino_predict(compiled_model, X_np)
    metrics = evaluate_numpy_predictions(preds, y_np)
    metrics.update(validate_openvino_parity(preds, reference_preds, metrics, reference_metrics))
    return {
        **metrics,
        **measure_openvino_latency(compiled_model, X_np, 1, LATENCY_WARMUP, LATENCY_RUNS_B1),
        **measure_openvino_latency(compiled_model, X_np, 64, LATENCY_WARMUP, LATENCY_RUNS_B64),
    }


def record_runtime_benchmark_skip(rows: list, model_name: str, runtime: str,
                                  variant: str, reason: str) -> None:
    rows.append({
        "model_name": model_name,
        "runtime": runtime,
        "variant": variant,
        "status": "skipped",
        "skip_reason": reason,
        "accuracy": None,
        "macro_precision": None,
        "macro_recall": None,
        "macro_f1": None,
        "serialized_size_kb": None,
        "latency_p50_ms_b1": None,
        "latency_p95_ms_b1": None,
        "latency_p99_ms_b1": None,
        "latency_p50_ms_b64": None,
        "latency_p95_ms_b64": None,
        "latency_p99_ms_b64": None,
        "throughput_samples_per_s_b1": None,
        "throughput_samples_per_s_b64": None,
        "artifact_path": None,
    })


def append_runtime_benchmark_row(rows: list, model_name: str, runtime: str,
                                 variant: str, metrics: dict,
                                 artifact_path: Path, serialized_size_kb: float) -> None:
    row = {
        "model_name": model_name,
        "runtime": runtime,
        "variant": variant,
        "status": "ok",
        "skip_reason": None,
        "accuracy": metrics["accuracy"],
        "macro_precision": metrics["macro_precision"],
        "macro_recall": metrics["macro_recall"],
        "macro_f1": metrics["macro_f1"],
        "serialized_size_kb": serialized_size_kb,
        "artifact_path": str(artifact_path),
    }
    for key, value in metrics.items():
        if key.startswith("latency_") or key.startswith("throughput_") or key.startswith("openvino_"):
            row[key] = value
    rows.append(row)

## Run ONNX Runtime and OpenVINO benchmarks from existing artifacts

In [6]:
def run_existing_artifact_runtime_benchmarks() -> tuple:
    optional = try_import_optional_deployment_modules()
    optional_dependency_status = optional["optional_dependency_status"]
    rows = []
    onnx_ready = ENABLE_ONNX_BENCHMARKS and optional_dependency_status.get("onnx_available")
    ort_ready = onnx_ready and optional_dependency_status.get("onnxruntime_available")
    ov_ready = ENABLE_OPENVINO_BENCHMARKS and onnx_ready and optional_dependency_status.get("openvino_available")

    for model_name in DEPLOYMENT_BENCHMARK_MODELS:
        try:
            model = load_existing_student_model(model_name)
        except Exception as exc:
            for runtime, variant in [
                ("onnxruntime", "onnx_fp32"),
                ("onnxruntime", "onnx_dynamic_int8"),
                ("openvino", "openvino_fp32_from_onnx"),
            ]:
                record_runtime_benchmark_skip(rows, model_name, runtime, variant,
                                              f"artifact_load_failed: {type(exc).__name__}: {exc}")
            continue

        onnx_path = artifact_dir / f"{model_name}.onnx"
        if not onnx_ready:
            for runtime, variant in [
                ("onnxruntime", "onnx_fp32"),
                ("onnxruntime", "onnx_dynamic_int8"),
                ("openvino", "openvino_fp32_from_onnx"),
            ]:
                record_runtime_benchmark_skip(rows, model_name, runtime, variant, "missing_dependency")
            continue

        try:
            onnx_size_kb = export_student_to_onnx(model, onnx_path)
        except Exception as exc:
            for runtime, variant in [
                ("onnxruntime", "onnx_fp32"),
                ("onnxruntime", "onnx_dynamic_int8"),
                ("openvino", "openvino_fp32_from_onnx"),
            ]:
                record_runtime_benchmark_skip(rows, model_name, runtime, variant,
                                              f"onnx_export_failed: {type(exc).__name__}: {exc}")
            continue

        onnx_fp32_metrics = None
        onnx_fp32_preds = None
        if ort_ready:
            try:
                metrics, preds = benchmark_onnxruntime_model(
                    onnx_path, X_test_np, y_test_np, return_preds=True
                )
                onnx_fp32_metrics = metrics
                onnx_fp32_preds = preds
                append_runtime_benchmark_row(rows, model_name, "onnxruntime", "onnx_fp32",
                                             metrics, onnx_path, onnx_size_kb)
            except Exception as exc:
                record_runtime_benchmark_skip(rows, model_name, "onnxruntime", "onnx_fp32",
                                              f"benchmark_failed: {type(exc).__name__}: {exc}")

            q_path = artifact_dir / f"{model_name}_dynamic_int8.onnx"
            try:
                q_size_kb = quantize_onnx_dynamic(onnx_path, q_path)
                metrics = benchmark_onnxruntime_model(q_path, X_test_np, y_test_np)
                append_runtime_benchmark_row(rows, model_name, "onnxruntime", "onnx_dynamic_int8",
                                             metrics, q_path, q_size_kb)
            except Exception as exc:
                record_runtime_benchmark_skip(rows, model_name, "onnxruntime", "onnx_dynamic_int8",
                                              f"quant_or_benchmark_failed: {type(exc).__name__}: {exc}")
        else:
            record_runtime_benchmark_skip(rows, model_name, "onnxruntime", "onnx_fp32", "missing_dependency")
            record_runtime_benchmark_skip(rows, model_name, "onnxruntime", "onnx_dynamic_int8", "missing_dependency")

        if ov_ready:
            if onnx_fp32_preds is None or onnx_fp32_metrics is None:
                record_runtime_benchmark_skip(
                    rows, model_name, "openvino", "openvino_fp32_from_onnx",
                    "missing_onnxruntime_reference_for_parity"
                )
            else:
                try:
                    metrics = benchmark_openvino_model(
                        onnx_path, X_test_np, y_test_np,
                        reference_preds=onnx_fp32_preds,
                        reference_metrics=onnx_fp32_metrics,
                    )
                    append_runtime_benchmark_row(rows, model_name, "openvino", "openvino_fp32_from_onnx",
                                                 metrics, onnx_path, onnx_size_kb)
                except Exception as exc:
                    record_runtime_benchmark_skip(rows, model_name, "openvino", "openvino_fp32_from_onnx",
                                                  f"benchmark_failed: {type(exc).__name__}: {exc}")
        else:
            record_runtime_benchmark_skip(rows, model_name, "openvino", "openvino_fp32_from_onnx", "missing_dependency")
    return rows, optional_dependency_status


deployable_runtime_rows, optional_dependency_status = run_existing_artifact_runtime_benchmarks()
runtime_df = pd.DataFrame(deployable_runtime_rows)
runtime_csv = output_dir / "wsnds_existing_artifact_runtime_summary.csv"
runtime_df.to_csv(runtime_csv, index=False)

runtime_json = output_dir / "wsnds_existing_artifact_runtime_results.json"
with open(runtime_json, "w", encoding="utf-8") as handle:
    json.dump({
        "source_existing_deployment_output_dir": str(existing_output_dir),
        "model_artifacts": MODEL_ARTIFACTS,
        "optional_dependency_status": optional_dependency_status,
        "results": deployable_runtime_rows,
        "environment": {
            "python": sys.version,
            "platform": platform.platform(),
            "torch": torch.__version__,
            "numpy": np.__version__,
            "pandas": pd.__version__,
        },
    }, handle, indent=2)

print(f"Saved {runtime_csv}")
print(f"Saved {runtime_json}")
print(runtime_df[["model_name", "runtime", "variant", "status", "macro_f1", "latency_p50_ms_b1", "latency_p50_ms_b64"]].to_string(index=False))

C:\Users\btpUser\AppData\Local\Temp\ipykernel_33880\317833118.py:39: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(artifact_path, map_location="cpu")
C:\U

Saved C:\Users\btpUser\Desktop\Nishant_2023IMG_wct\CuKD-XAI\Final\wsnds_deployment_qat_outputs\runtime_from_existing_outputs\wsnds_existing_artifact_runtime_summary.csv
Saved C:\Users\btpUser\Desktop\Nishant_2023IMG_wct\CuKD-XAI\Final\wsnds_deployment_qat_outputs\runtime_from_existing_outputs\wsnds_existing_artifact_runtime_results.json
                 model_name     runtime                 variant status  macro_f1  latency_p50_ms_b1  latency_p50_ms_b64
        D_student_A_scratch onnxruntime               onnx_fp32     ok  0.906946             0.0267             0.03590
        D_student_A_scratch onnxruntime       onnx_dynamic_int8     ok  0.891391             0.0393             0.05000
        D_student_A_scratch    openvino openvino_fp32_from_onnx     ok  0.906946             0.1289             0.17015
     E_student_A_KD_from_RF onnxruntime               onnx_fp32     ok  0.917478             0.0275             0.03520
     E_student_A_KD_from_RF onnxruntime       onnx_dynamic_in